# Makine Öğrenmesi Lab 03 — Hoca Versiyonu
## Makine Öğrenmesi Deneylerinin Tasarımı ve Analizi

**Süre:** 75–90 dakika  
**Seviye:** Başlangıç  
**Kütüphane:** Harici kütüphane yok

### Öğrenme hedefleri
- Eğitim, doğrulama ve test verisinin görevini ayırt etmek
- Basit bir sınıflandırıcının performansını ölçmek
- Confusion matrix, Accuracy, Precision, Recall ve F1-score hesaplamak
- Dengesiz veride Accuracy'nin neden yanıltıcı olabileceğini görmek
- Basit cross-validation mantığını anlamak
- Modelleri adil biçimde karşılaştırmak
- Data leakage kavramını tanımak

> **Hoca notu:** Amaç sklearn fonksiyonlarını ezberletmek değil, deney tasarımının mantığını görünür hale getirmek.

# 0. Geçen Haftadan Kısa Tekrar
Geçen hafta ortalama, varyans, standart sapma, vektörler, parametre ve basit doğrusal model mantığını gördük.

Bu hafta şu soruya cevap arıyoruz:

> **Bir modelin gerçekten iyi olduğunu nasıl anlarız?**

In [ ]:
gercek_deger = 80
tahmin = 76
hata = gercek_deger - tahmin
print("Gerçek değer:", gercek_deger)
print("Tahmin:", tahmin)
print("Hata:", hata)

# 1. Training Set ve Test Set

Bir veri kümesini en basit haliyle iki parçaya ayırabiliriz:

- **Training set:** Modelin öğrendiği veriler
- **Test set:** Modelin daha önce görmediği, final değerlendirmede kullanılan veriler

```text
Tüm Veri
   |
   +---- Training
   |
   +---- Test
```

İyi bir model yalnızca eğitim verisini ezberlememeli; yeni örneklerde de doğru tahmin yapabilmelidir. Buna **genelleme** denir.

In [ ]:
# [çalışma_saati, geçti_mi]
# 1 = geçti, 0 = kaldı
veri = [
    [1, 0], [2, 0], [2.5, 0], [3, 0],
    [4, 1], [4.5, 1], [5, 1], [6, 1],
    [7, 1], [8, 1]
]

egitim_verisi = veri[:8]
test_verisi = veri[8:]

print("Toplam örnek:", len(veri))
print("Eğitim örneği:", len(egitim_verisi))
print("Test örneği:", len(test_verisi))

## Sınıfa sor
1. Model training verisini görür mü? **Evet**
2. Model test verisini eğitim sırasında görmeli mi? **Hayır**
3. Test setinin görevi nedir? **Görülmemiş veride performansı ölçmek.**

# 2. Neden Aynı Veriyle Eğitim ve Test Yapmıyoruz?

Bir öğrenci sınavdan önce bütün soruların cevaplarını ezberlerse, aynı sorularla yapılan sınavda yüksek puan alabilir. Bu, konuyu gerçekten öğrendiği anlamına gelmez.

Makine öğrenmesinde de model eğitim örneklerini ezberleyebilir. Bu nedenle değerlendirme için modelin görmediği veriye ihtiyaç duyarız.

> İleride bunu **overfitting (aşırı öğrenme)** başlığı altında ayrıntılı göreceğiz.

# 3. Çok Basit Bir Sınıflandırıcı

Kuralımız:

```text
çalışma saati >= eşik  →  Geçti (1)
çalışma saati < eşik   →  Kaldı (0)
```

Buradaki **eşik**, modelimizin ayarlanabilir parametresi gibi düşünülebilir.

In [ ]:
def tahmin_yap(calisma_saati, esik):
    if calisma_saati >= esik:
        return 1
    else:
        return 0

print("5 saat, eşik 4:", tahmin_yap(5, 4))
print("2 saat, eşik 4:", tahmin_yap(2, 4))

# 4. Accuracy — Doğruluk

\[
Accuracy = rac{Doğru\ Tahmin\ Sayısı}{Toplam\ Tahmin\ Sayısı}
\]

10 örneğin 8'i doğruysa Accuracy = 8/10 = 0.80 = %80.

In [ ]:
gercek = [1, 0, 1, 1, 0]
tahminler = [1, 0, 0, 1, 0]

dogru = 0
for i in range(len(gercek)):
    if gercek[i] == tahminler[i]:
        dogru += 1

accuracy = dogru / len(gercek)
print("Doğru tahmin:", dogru)
print("Toplam:", len(gercek))
print("Accuracy:", accuracy)
print("Yüzde:", accuracy * 100)

## Alıştırma 1

```python
gercek = [1, 1, 0, 1, 0, 0]
tahminler = [1, 0, 0, 1, 1, 0]
```

Accuracy'yi hesaplayın.

In [ ]:
# Alıştırma 1 — Hoca çözümü
gercek = [1, 1, 0, 1, 0, 0]
tahminler = [1, 0, 0, 1, 1, 0]

dogru = 0
for i in range(len(gercek)):
    if gercek[i] == tahminler[i]:
        dogru += 1

accuracy = dogru / len(gercek)
print("Doğru tahmin:", dogru)
print("Accuracy:", accuracy)
print("Yüzde:", accuracy * 100)

# 5. Confusion Matrix

İkili sınıflandırmada dört durum vardır:

| Gerçek | Tahmin | Adı |
|---|---|---|
| 1 | 1 | True Positive (TP) |
| 0 | 0 | True Negative (TN) |
| 0 | 1 | False Positive (FP) |
| 1 | 0 | False Negative (FN) |

- **TP:** Pozitifi doğru bulduk
- **TN:** Negatifi doğru bulduk
- **FP:** Negatifti ama pozitif dedik
- **FN:** Pozitifti ama negatif dedik

In [ ]:
gercek = [1, 1, 0, 0, 1, 0]
tahminler = [1, 0, 0, 1, 1, 0]

TP = TN = FP = FN = 0

for i in range(len(gercek)):
    if gercek[i] == 1 and tahminler[i] == 1:
        TP += 1
    elif gercek[i] == 0 and tahminler[i] == 0:
        TN += 1
    elif gercek[i] == 0 and tahminler[i] == 1:
        FP += 1
    elif gercek[i] == 1 and tahminler[i] == 0:
        FN += 1

print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)

# 6. Precision

Precision şu soruya cevap verir:

> **Model pozitif dediğinde, ne kadar güvenebiliriz?**

\[
Precision = rac{TP}{TP + FP}
\]

In [ ]:
TP = 2
FP = 1
precision = TP / (TP + FP)
print("Precision:", precision)

# 7. Recall

Recall şu soruya cevap verir:

> **Gerçek pozitiflerin ne kadarını bulabildik?**

\[
Recall = rac{TP}{TP + FN}
\]

In [ ]:
TP = 2
FN = 1
recall = TP / (TP + FN)
print("Recall:", recall)

## Precision ve Recall farkı

**Spam filtresi:** Yanlışlıkla önemli e-postayı spam yapmak istemeyebiliriz → precision önemli olabilir.

**Hastalık taraması:** Gerçek hastaları kaçırmamak isteriz → recall önemli olabilir.

Bu yüzden değerlendirme metriği probleme göre seçilir.

# 8. F1-score

Precision ve Recall'u tek bir değerde dengelemek için kullanılabilir:

\[
F1 = 2 	imes rac{Precision 	imes Recall}{Precision + Recall}
\]

In [ ]:
precision = 2 / 3
recall = 2 / 3
f1 = 2 * (precision * recall) / (precision + recall)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

## Alıştırma 2 — Bütün metrikleri hesapla

```text
TP = 30
TN = 50
FP = 10
FN = 10
```

- Accuracy
- Precision
- Recall
- F1

In [ ]:
# Alıştırma 2 — Hoca çözümü
TP = 30
TN = 50
FP = 10
FN = 10

toplam = TP + TN + FP + FN
accuracy = (TP + TN) / toplam
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * (precision * recall) / (precision + recall)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

# 9. Accuracy Neden Yanıltıcı Olabilir?

100 kişinin yalnızca 5'i hasta olsun. Model herkese **sağlıklı** desin.

95 kişiyi doğru tahmin eder:

\[
Accuracy = 95/100 = 95\%
\]

Ama model **tek bir hastayı bile bulamamıştır.**

Bu, özellikle dengesiz sınıflarda Accuracy'nin tek başına yeterli olmadığını gösterir.

In [ ]:
toplam_kisi = 100
dogru_tahmin = 95
accuracy = dogru_tahmin / toplam_kisi
print("Accuracy:", accuracy)
print("Accuracy yüzde:", accuracy * 100)

# 10. Model Karşılaştırması

İki modelimiz olsun:

- Model A: eşik = 4
- Model B: eşik = 5

Aynı test verisi üzerinde karşılaştıracağız.

In [ ]:
test_verisi = [
    [2, 0], [3.5, 0], [4, 1], [4.5, 1],
    [5, 1], [6, 1], [7, 1], [3, 0]
]

def accuracy_hesapla(veri, esik):
    dogru = 0
    for satir in veri:
        saat = satir[0]
        gercek = satir[1]
        tahmin = tahmin_yap(saat, esik)
        if tahmin == gercek:
            dogru += 1
    return dogru / len(veri)

print("Model A, eşik 4:", accuracy_hesapla(test_verisi, 4))
print("Model B, eşik 5:", accuracy_hesapla(test_verisi, 5))

## Adil model karşılaştırması

Modelleri karşılaştırırken mümkün olduğunca:
- aynı veri,
- aynı veri ayrımı,
- aynı metrikler,
- aynı değerlendirme koşulları

kullanılmalıdır.

# 11. Validation Set

Model seçmek için test setini sürekli kullanmak doğru değildir.

Bu yüzden çoğu deneyde üçüncü bir bölüm kullanılır:

```text
Tüm Veri
   |
   +---- Training
   +---- Validation
   +---- Test
```

- **Training:** Modeli öğren
- **Validation:** Model/parametre seç
- **Test:** Final değerlendirme

Örnek oran: %60 / %20 / %20. Bu oranlar sabit değildir.

In [ ]:
train = 60
validation = 20
test = 20
print("Training:", train)
print("Validation:", validation)
print("Test:", test)
print("Toplam:", train + validation + test)

# 12. Data Leakage — Veri Sızıntısı

**Data leakage**, modelin eğitim sırasında normalde erişmemesi gereken bir bilgiye erişmesidir.

Örneğin bir kredinin geri ödenip ödenmeyeceğini tahmin ederken, kredinin kapanmasından sonra oluşan bilgiyi girdi olarak kullanmak gerçek hayatta mümkün değildir.

Ayrıca test setine defalarca bakıp modeli ona göre ayarlamak da test setini tarafsız bir final ölçüm aracı olmaktan çıkarır.

## Sınıfa soru
Hangisi doğru?

**A)** Test verisini model seçerken sürekli kullanmak  
**B)** Test verisini final değerlendirme için saklamak

**Cevap: B**

# 13. Cross-Validation Mantığı

Veri az olduğunda tek bir train-validation ayrımı şanslı veya şanssız olabilir.

Örneğin 4-fold cross-validation:

```text
Fold 1: VALIDATION | TRAIN | TRAIN | TRAIN
Fold 2: TRAIN | VALIDATION | TRAIN | TRAIN
Fold 3: TRAIN | TRAIN | VALIDATION | TRAIN
Fold 4: TRAIN | TRAIN | TRAIN | VALIDATION
```

Sonra skorların ortalamasını alırız.

In [ ]:
fold_skorlari = [0.80, 0.75, 0.85, 0.80]
toplam = 0
for skor in fold_skorlari:
    toplam += skor
ortalama = toplam / len(fold_skorlari)
print("Fold skorları:", fold_skorlari)
print("Ortalama CV skoru:", ortalama)

# 14. Basit Cross-Validation Uygulaması

Gerçek k-fold eğitimini sadeleştirerek, farklı veri parçalarında modelin skorunun nasıl değiştiğini görelim.

In [ ]:
veri = [
    [1, 0], [2, 0], [2.5, 0], [3, 0],
    [4, 1], [4.5, 1], [5, 1], [6, 1],
    [7, 1], [8, 1], [3.5, 0], [5.5, 1]
]

fold1 = veri[0:4]
fold2 = veri[4:8]
fold3 = veri[8:12]
foldlar = [fold1, fold2, fold3]

print("Fold 1:", fold1)
print("Fold 2:", fold2)
print("Fold 3:", fold3)

In [ ]:
esik = 4
fold_skorlari = []

for fold in foldlar:
    skor = accuracy_hesapla(fold, esik)
    fold_skorlari.append(skor)

ortalama_skor = sum(fold_skorlari) / len(fold_skorlari)
print("Fold skorları:", fold_skorlari)
print("Ortalama skor:", ortalama_skor)

> **Hoca notu:** Gerçek k-fold'da her turda bir fold validation, kalan fold'lar training olur. Buradaki model eğitim aşaması çok basit olduğu için odak değerlendirme mantığıdır.

# 15. Cross-Validation ile Model Karşılaştırma

- Model A: eşik = 3
- Model B: eşik = 4
- Model C: eşik = 5

Ortalama skoru en yüksek olanı seçelim.

In [ ]:
def cv_ortalama(foldlar, esik):
    skorlar = []
    for fold in foldlar:
        skorlar.append(accuracy_hesapla(fold, esik))
    return sum(skorlar) / len(skorlar), skorlar

esikler = [3, 4, 5]
en_iyi_esik = None
en_iyi_skor = -1

for esik in esikler:
    ortalama, skorlar = cv_ortalama(foldlar, esik)
    print("Eşik:", esik, "Skorlar:", skorlar, "Ortalama:", ortalama)
    if ortalama > en_iyi_skor:
        en_iyi_skor = ortalama
        en_iyi_esik = esik

print("En iyi eşik:", en_iyi_esik)
print("En iyi CV skoru:", en_iyi_skor)

# 16. Mini Proje — Öğrenci Başarı Tahmini

Aday modeller:
- eşik = 3
- eşik = 4
- eşik = 5

Önce **validation** üzerinde model seçeceğiz. Sonra seçilen modeli **test** üzerinde yalnızca bir kez final olarak değerlendireceğiz.

In [ ]:
validation_verisi = [
    [1.5, 0], [2.5, 0], [3.5, 0],
    [4, 1], [4.5, 1], [6, 1]
]

test_verisi = [
    [2, 0], [3, 0], [4.2, 1], [5, 1], [7, 1]
]

aday_esikler = [3, 4, 5]

In [ ]:
# 1. adım: Validation üzerinde seçim
en_iyi_esik = None
en_iyi_validation_skoru = -1

for esik in aday_esikler:
    skor = accuracy_hesapla(validation_verisi, esik)
    print("Eşik:", esik, "Validation Accuracy:", skor)
    if skor > en_iyi_validation_skoru:
        en_iyi_validation_skoru = skor
        en_iyi_esik = esik

print("Seçilen eşik:", en_iyi_esik)
print("Validation skoru:", en_iyi_validation_skoru)

### Burada öğrenciye sor
**Neden test verisine henüz bakmadık?**

Çünkü test setini model seçimi için kullanırsak, artık tarafsız final değerlendirme setimiz kalmaz.

In [ ]:
# 2. adım: Test üzerinde yalnızca final değerlendirme
test_accuracy = accuracy_hesapla(test_verisi, en_iyi_esik)
print("Seçilen eşik:", en_iyi_esik)
print("Final Test Accuracy:", test_accuracy)

# 17. Mini Projede Confusion Matrix ve Tüm Metrikler

In [ ]:
TP = TN = FP = FN = 0

for satir in test_verisi:
    saat = satir[0]
    gercek = satir[1]
    tahmin = tahmin_yap(saat, en_iyi_esik)

    if gercek == 1 and tahmin == 1:
        TP += 1
    elif gercek == 0 and tahmin == 0:
        TN += 1
    elif gercek == 0 and tahmin == 1:
        FP += 1
    elif gercek == 1 and tahmin == 0:
        FN += 1

print("TP:", TP)
print("TN:", TN)
print("FP:", FP)
print("FN:", FN)

In [ ]:
toplam = TP + TN + FP + FN
accuracy = (TP + TN) / toplam
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

# 18. Deney Sonucu Nasıl Raporlanır?

Sadece:

```text
Accuracy = 0.83
```

demek çoğu zaman yeterli değildir.

Daha iyi örnek:

```text
Model: Eşik tabanlı sınıflandırıcı
Model seçimi: Validation set
Final değerlendirme: Test set
Accuracy: ...
Precision: ...
Recall: ...
F1: ...
```

Ayrıca veri sayısı, veri ayrımı ve model parametreleri de belirtilmelidir.

# 19. Ders Sonu Hızlı Sorular

1. **Training set ne için?** → Modelin öğrenmesi için.
2. **Validation set ne için?** → Model/parametre seçmek için.
3. **Test set ne için?** → Final değerlendirme için.
4. **Accuracy her zaman yeterli mi?** → Hayır.
5. **Precision neyi ölçer?** → Pozitif dediğimiz örneklerin ne kadarının gerçekten pozitif olduğunu.
6. **Recall neyi ölçer?** → Gerçek pozitiflerin ne kadarını bulduğumuzu.
7. **Cross-validation neden yapılır?** → Tek bir ayrımın etkisini azaltmak ve daha güvenilir değerlendirme yapmak için.
8. **Data leakage nedir?** → Modelin normalde erişmemesi gereken bilgiye erişmesi.

# 20. 90 Dakikalık Hoca Planı

| Süre | Konu |
|---:|---|
| 0–8 dk | Geçen hafta tekrarı ve deney kavramı |
| 8–18 dk | Training-test ayrımı ve genelleme |
| 18–28 dk | Basit sınıflandırıcı ve Accuracy |
| 28–43 dk | Confusion matrix |
| 43–55 dk | Precision, Recall, F1 |
| 55–63 dk | Dengesiz veri problemi |
| 63–72 dk | Validation ve data leakage |
| 72–82 dk | Cross-validation |
| 82–90 dk | Mini proje ve sonuç yorumu |

### 60–70 dakikaya düşürmek gerekirse
- Cross-validation kodunu tek örnekle göster.
- CV model karşılaştırmasını ödev bırak.
- Mini projede F1 hesabını ev ödevi yap.

### Bu haftanın üç ana mesajı
1. **Training başarısı tek başına yeterli değildir.**
2. **Test verisi görülmemiş veriyi temsil eder.**
3. **Bir modeli tek bir metrikle değerlendirmek çoğu zaman doğru değildir.**

# 21. Sonraki Haftaya Köprü

Bir sonraki hafta **Boyut İndirgeme** konusuna geçeceğiz.

Hatırlanması gerekenler:
- Vektör
- Ortalama
- Varyans
- Birden fazla özellik
- Özelliklerin birbirleriyle ilişkisi

Bunlar PCA gibi yöntemlerin temelini hazırlayacak.